In [2]:
from training.dataset_classes.text_datasets import TextClassificationDataset
import pandas as pd
import numpy as np
from pathlib import Path
from itertools import product
from glob import glob

### 1. Create train dataset

In [2]:
# data_path = '/app/datasets/text_datasets/yahoo_answers_csv/in_distribution_train.csv'
# df = pd.read_csv(
#     data_path,           # or 'your_file.csv'
#     header=None,
#     usecols=[0, 2],           # keep only column 0 (first) and column 2 (third)
#     names=['class', 'text'],   # rename them as needed
#     quotechar='"'
# )
# df['text'] = df['text'].str.strip('"\'')
# # Re-check
# print("Remaining leading/trailing quotes:",
#       (df['text'].str.startswith('"') | df['text'].str.endswith('"') |
#        df['text'].str.startswith("'") | df['text'].str.endswith("'")).sum())
# unique_classes = sorted(np.unique(df['class'].values)) 
# class_to_idx = {cls: idx for idx, cls in enumerate(unique_classes)}
# df['class'] = df['class'].map(class_to_idx)

### 2. Create an OSR protocol for text

In [3]:
# data_path = "/app/datasets/text_datasets/yahoo_answers_csv/in_distribution_test.csv"
# df = pd.read_csv(
#     data_path,  # or 'your_file.csv'
#     header=None,
#     usecols=[0, 2],  # keep only column 0 (first) and column 2 (third)
#     names=["class", "text"],  # rename them as needed
#     quotechar='"',
# )
# df["text"] = df["text"].str.strip("\"'")

In [4]:
#np.unique(df["class"].values, return_counts=True)

##### Protocol construction
1. Each sample is its own template
2. I have to split in-distribution test set into gallery and probes.  
I will randomly choose 2% of in-distribution samples from each class to construct gallery templates. 
3. I will use out-of-disribution samples as it is

In [5]:
#np.where(df["class"].values == 2)[0]

In [6]:
# from training.dataset_classes.text_datasets import TextPredictionDataset
# ds = TextPredictionDataset('/app/datasets/text-ident/yahoo_answers')
# ds[1]

In [7]:
# ident_ds_dir = Path("/app/datasets/text-ident")
# ident_ds_dir.mkdir(exist_ok=True)

# # list protocols
# protools_path = "/app/datasets/text_datasets"
# protools_paths = list(glob(protools_path + "/*_csv"))

# # create meta files
# template_idx_shift = 10000  # shift to get unique template ids
# gallery_template_size_fraction = 1e-2
# min_gallery_template_size = 50
# rng = np.random.default_rng(32)

# for protocol_path in protools_paths:
#     protocol_path = Path(protocol_path)
#     print(protocol_path)
#     ds_name = (protocol_path.parts[-1]).lower()[:-4]
#     meta_path = ident_ds_dir / ds_name / "meta"
#     meta_path.mkdir(exist_ok=True, parents=True)

#     ossi_info_path = protocol_path
#     df_in_disribution = pd.read_csv(
#         ossi_info_path / "in_distribution_test.csv",
#         header=None,
#         usecols=[0, 2],
#         names=["class", "text"],
#         quotechar='"',
#     )
#     df_in_disribution["text"] = df_in_disribution["text"].str.strip("\"'")

#     df_ood = pd.read_csv(
#         ossi_info_path / "out_distribution_test.csv",
#         header=None,
#         usecols=[0, 2],
#         names=["class", "text"],
#         quotechar='"',
#     )
#     df_ood["text"] = df_ood["text"].str.strip("\"'")

#     known_classes, known_count = np.unique(df_in_disribution["class"].values, return_counts=True)
#     gallery_template_sizes = np.max(
#         np.stack(
#             [
#                 np.array([min_gallery_template_size] * known_classes.shape[0]),
#                 (
#                     gallery_template_size_fraction
#                     * known_count
#                 ).astype("int"),
#             ],
#             axis=1,
#         ),
#         axis=1,
#     )
#     # save texts into files
#     known_save_dir = ident_ds_dir / ds_name / 'known_texts'
#     known_save_dir.mkdir(exist_ok=True)
#     for i, text in enumerate(df_in_disribution['text'].values):
#         with open(known_save_dir / f'{i}.txt', 'w') as fd:
#             fd.write(text)
#     unknown_save_dir = ident_ds_dir / ds_name / 'unknown_texts'
#     unknown_save_dir.mkdir(exist_ok=True)
#     for i, text in enumerate(df_ood['text'].values):
#         with open(unknown_save_dir / f'{i}.txt', 'w') as fd:
#             fd.write(text)

#     # select gallery and probe templates
#     gallery_paths = []
#     gallery_ids = []
#     probe_paths = []
#     probe_ids = []
#     for i, known_class in enumerate(known_classes):
#         sample_of_class = np.where(df_in_disribution["class"].values == known_class)[0]
#         gallery_sample_idx = rng.choice(sample_of_class, size=gallery_template_sizes[i], replace=False)
#         probe_sample_idx = set(sample_of_class) - set(gallery_sample_idx)
#         gallery_paths.append([f'known_texts/{sample_id}.txt' for sample_id in gallery_sample_idx])
#         gallery_ids.append(([known_class] * gallery_template_sizes[i]))
#         probe_paths.append([f'known_texts/{sample_id}.txt' for sample_id in np.array(list(probe_sample_idx))])
#         probe_ids.append([known_class] * len(probe_sample_idx))
#     gallery_paths = np.concatenate(gallery_paths)
#     gallery_ids = np.concatenate(gallery_ids)
#     probe_paths = np.concatenate(probe_paths)
#     probe_ids = np.concatenate(probe_ids)

#     probe_paths = np.concatenate([probe_paths, [f'unknown_texts/{sample_id}.txt' for sample_id in range(len(df_ood['text'].values))]])
#     probe_ids = np.concatenate([probe_ids, df_ood['class'].values])
    
#     probe_template_ids = np.arange(len(probe_ids))+template_idx_shift
#     # # create tid/mid file
#     text_paths = np.concatenate([gallery_paths,probe_paths])
#     ids = np.concatenate([gallery_ids, probe_ids])
#     tids = np.concatenate([gallery_ids, probe_template_ids])
#     mids = np.arange(len(ids))
#     out_file_tid_mid = meta_path / Path(f"{ds_name}_face_tid_mid.txt")
#     with open(out_file_tid_mid, "w") as fd:
#         for name, tid, sid, mid in zip(text_paths, tids, ids, mids):
#             fd.write(f"{name} {tid} {mid} {sid}\n")

#     # # create gallery and probe meta files
#     out_file_probe = meta_path / Path(f"{ds_name}_1N_probe_mixed.csv")
#     out_file_gallery = meta_path / Path(f"{ds_name}_1N_gallery_G1.csv")

#     assert len(gallery_ids) + len(probe_ids) == len(text_paths)
#     probe = pd.DataFrame(
#         {
#             "TEMPLATE_ID": probe_template_ids,
#             "SUBJECT_ID": probe_ids,
#             "FILENAME": probe_paths,
#         }
#     )
#     gallery = pd.DataFrame(
#         {
#             "TEMPLATE_ID": gallery_ids,
#             "SUBJECT_ID": gallery_ids,
#             "FILENAME": gallery_paths,
#         }
#     )

#     probe.to_csv(out_file_probe, sep=",", index=False)
#     gallery.to_csv(out_file_gallery, sep=",", index=False)

### Validation sets

In [ ]:
# ident_ds_dir = Path("/app/datasets/text-ident-val")
# ident_ds_dir.mkdir(exist_ok=True)

# # list protocols
# protools_path = "/app/datasets/text_datasets"
# protools_paths = list(glob(protools_path + "/*_csv"))

# # create meta files
# template_idx_shift = 10000  # shift to get unique template ids
# gallery_template_size_fraction = 1e-2
# min_gallery_template_size = 50
# rng = np.random.default_rng(32)

# for protocol_path in protools_paths:
#     protocol_path = Path(protocol_path)
#     print(protocol_path)
#     ds_name = (protocol_path.parts[-1]).lower()[:-4]
#     meta_path = ident_ds_dir / ds_name / "meta"
#     meta_path.mkdir(exist_ok=True, parents=True)

#     ossi_info_path = protocol_path
#     df_in_disribution = pd.read_csv(
#         ossi_info_path / "in_distribution_train.csv",
#         header=None,
#         usecols=[0, 2],
#         names=["class", "text"],
#         quotechar='"',
#     )
#     df_in_disribution_test = pd.read_csv(
#         ossi_info_path / "in_distribution_test.csv",
#         header=None,
#         usecols=[0, 2],
#         names=["class", "text"],
#         quotechar='"',
#     )
#     df_in_disribution["text"] = df_in_disribution["text"].str.strip("\"'")
#     df_ood = pd.read_csv(
#         ossi_info_path / "out_distribution_train.csv",
#         header=None,
#         usecols=[0, 2],
#         names=["class", "text"],
#         quotechar='"',
#     )
#     df_ood_test = pd.read_csv(
#         ossi_info_path / "out_distribution_test.csv",
#         header=None,
#         usecols=[0, 2],
#         names=["class", "text"],
#         quotechar='"',
#     )
#     df_ood["text"] = df_ood["text"].str.strip("\"'")

#     known_classes, known_count_test = np.unique(df_in_disribution_test["class"].values, return_counts=True)
#     gallery_template_sizes = np.max(
#         np.stack(
#             [
#                 np.array([min_gallery_template_size] * known_classes.shape[0]),
#                 (
#                     gallery_template_size_fraction
#                     * known_count_test
#                 ).astype("int"),
#             ],
#             axis=1,
#         ),
#         axis=1,
#     )
#     # save texts into files
#     known_save_dir = ident_ds_dir / ds_name / 'known_texts'
#     known_save_dir.mkdir(exist_ok=True)

    
#     ood_texts = df_ood['text'].values
#     ood_class = df_ood['class'].values
#     ood_ids = rng.choice(np.arange(len(ood_texts)), size=len(df_ood_test['text'].values), replace=False)
#     unknown_save_dir = ident_ds_dir / ds_name / 'unknown_texts'
#     unknown_save_dir.mkdir(exist_ok=True)
#     for i, text in enumerate(ood_texts[ood_ids]):
#         with open(unknown_save_dir / f'{i}.txt', 'w') as fd:
#             fd.write(text)

#     # select gallery and probe templates
#     gallery_paths = []
#     gallery_ids = []
#     probe_paths = []
#     probe_ids = []
#     for i, known_class in enumerate(known_classes):
#         sample_of_class = np.where(df_in_disribution["class"].values == known_class)[0]
#         # sample the same number of in distributions sample as in test
#         sample_of_class = rng.choice(sample_of_class, known_count_test[i], replace=False)
#         for j, text in zip(sample_of_class, df_in_disribution['text'].values[sample_of_class]):
#             with open(known_save_dir / f'{j}.txt', 'w') as fd:
#                 fd.write(text)
#         gallery_sample_idx = rng.choice(sample_of_class, size=gallery_template_sizes[i], replace=False)
#         probe_sample_idx = set(sample_of_class) - set(gallery_sample_idx)
#         gallery_paths.append([f'known_texts/{sample_id}.txt' for sample_id in gallery_sample_idx])
#         gallery_ids.append(([known_class] * gallery_template_sizes[i]))
#         probe_paths.append([f'known_texts/{sample_id}.txt' for sample_id in np.array(list(probe_sample_idx))])
#         probe_ids.append([known_class] * len(probe_sample_idx))
#     gallery_paths = np.concatenate(gallery_paths)
#     gallery_ids = np.concatenate(gallery_ids)
#     probe_paths = np.concatenate(probe_paths)
#     probe_ids = np.concatenate(probe_ids)



#     probe_paths = np.concatenate([probe_paths, [f'unknown_texts/{sample_id}.txt' for sample_id in range(len(ood_ids))]])
#     probe_ids = np.concatenate([probe_ids, ood_class[ood_ids]])
    
#     probe_template_ids = np.arange(len(probe_ids))+template_idx_shift
#     # # create tid/mid file
#     text_paths = np.concatenate([gallery_paths,probe_paths])
#     ids = np.concatenate([gallery_ids, probe_ids])
#     tids = np.concatenate([gallery_ids, probe_template_ids])
#     mids = np.arange(len(ids))
#     out_file_tid_mid = meta_path / Path(f"{ds_name}_face_tid_mid.txt")
#     with open(out_file_tid_mid, "w") as fd:
#         for name, tid, sid, mid in zip(text_paths, tids, ids, mids):
#             fd.write(f"{name} {tid} {mid} {sid}\n")

#     # # create gallery and probe meta files
#     out_file_probe = meta_path / Path(f"{ds_name}_1N_probe_mixed.csv")
#     out_file_gallery = meta_path / Path(f"{ds_name}_1N_gallery_G1.csv")

#     assert len(gallery_ids) + len(probe_ids) == len(text_paths)
#     probe = pd.DataFrame(
#         {
#             "TEMPLATE_ID": probe_template_ids,
#             "SUBJECT_ID": probe_ids,
#             "FILENAME": probe_paths,
#         }
#     )
#     gallery = pd.DataFrame(
#         {
#             "TEMPLATE_ID": gallery_ids,
#             "SUBJECT_ID": gallery_ids,
#             "FILENAME": gallery_paths,
#         }
#     )

#     probe.to_csv(out_file_probe, sep=",", index=False)
#     gallery.to_csv(out_file_gallery, sep=",", index=False)

/app/datasets/text_datasets/dbpedia_csv
/app/datasets/text_datasets/yahoo_answers_csv
/app/datasets/text_datasets/agnews_csv


## Clinc150 (use embedding train set as the gallery)

In [4]:
from training.dataset_classes.text_datasets import Clinc150DataModule

clinc150_dataset = Clinc150DataModule('/app/datasets/clinc150/data_full.json')
clinc150_dataset.setup()

ds_type = 'val'
ident_ds_dir = Path(f"/app/datasets/clinc150_{ds_type}")
ident_ds_dir.mkdir(exist_ok=True)

# create meta files
template_idx_shift = 10000  # shift to get unique template ids
rng = np.random.default_rng(32)

meta_path = ident_ds_dir / "meta"
meta_path.mkdir(exist_ok=True, parents=True)

# init ds
if ds_type == 'val':
    ds = clinc150_dataset.val_dataset
elif ds_type == 'test':
    ds = clinc150_dataset.test_dataset

# save texts into files
# first save train texts, then val/test
text_save_dir = ident_ds_dir / 'texts'
text_save_dir.mkdir(exist_ok=True)

text_counter = 0
gallery_paths = []
gallery_ids = []
probe_paths = []
probe_ids = []
for i in range(len(clinc150_dataset.train_dataset)):
    gallery_ids.append(clinc150_dataset.train_dataset[i]['label'])
    gallery_paths.append(f'texts/{text_counter}.txt')
    with open(text_save_dir / f'{text_counter}.txt', 'w') as fd:
        fd.write(clinc150_dataset.train_dataset[i]['text'])
    text_counter+=1

for i in range(len(ds)):
    probe_ids.append(ds[i]['label'])
    probe_paths.append(f'texts/{text_counter}.txt')
    with open(text_save_dir / f'{text_counter}.txt', 'w') as fd:
        fd.write(ds[i]['text'])
    text_counter+=1

# select gallery and probe templates

gallery_paths = np.array(gallery_paths)
gallery_ids = np.array(gallery_ids)
probe_paths = np.array(probe_paths)
probe_ids = np.array(probe_ids)


probe_template_ids = np.arange(len(probe_ids)) + template_idx_shift
# # create tid/mid file
text_paths = np.concatenate([gallery_paths,probe_paths])
ids = np.concatenate([gallery_ids, probe_ids])
tids = np.concatenate([gallery_ids, probe_template_ids])
mids = np.arange(len(ids))
if ds_type == 'val':
    name_appendix = "_val"
else:
    name_appendix = ""
out_file_tid_mid = meta_path / Path(f"clinc150{name_appendix}_face_tid_mid.txt")
with open(out_file_tid_mid, "w") as fd:
    for name, tid, sid, mid in zip(text_paths, tids, ids, mids):
        fd.write(f"{name} {tid} {mid} {sid}\n")

# # create gallery and probe meta files
out_file_probe = meta_path / Path(f"clinc150{name_appendix}_1N_probe_mixed.csv")
out_file_gallery = meta_path / Path(f"clinc150{name_appendix}_1N_gallery_G1.csv")

assert len(gallery_ids) + len(probe_ids) == len(text_paths)
probe = pd.DataFrame(
    {
        "TEMPLATE_ID": probe_template_ids,
        "SUBJECT_ID": probe_ids,
        "FILENAME": probe_paths,
    }
)
gallery = pd.DataFrame(
    {
        "TEMPLATE_ID": gallery_ids,
        "SUBJECT_ID": gallery_ids,
        "FILENAME": gallery_paths,
    }
)

probe.to_csv(out_file_probe, sep=",", index=False)
gallery.to_csv(out_file_gallery, sep=",", index=False)

In [3]:
# copy_embs
val_embs_dir = Path(f"/app/datasets/clinc150_val") / 'embeddings'
val_embs_dir.mkdir(exist_ok=True)

test_embs_dir = Path(f"/app/datasets/clinc150_test") / 'embeddings'
test_embs_dir.mkdir(exist_ok=True)

train_embs = np.load('/app/cache/features/clinc150_train_embs.npz')
val_embs = np.load('/app/cache/features/clinc150_val_embs.npz')
test_embs = np.load('/app/cache/features/clinc150_test_embs.npz')

np.savez(val_embs_dir / 'scf_embs_clinc150_val.npz', embs=np.concatenate([train_embs['embs'], val_embs['embs']], axis=0),
         unc = np.concatenate([train_embs['unc'], val_embs['unc']], axis=0))
np.savez(test_embs_dir / 'scf_embs_clinc150.npz', embs=np.concatenate([train_embs['embs'], test_embs['embs']], axis=0),
         unc = np.concatenate([train_embs['unc'], test_embs['unc']], axis=0))

In [24]:
train_embs['embs'].shape, train_embs['unc'].shape

((15000, 768), (15000, 1))